# Resonator - Route C

Build semantic geometry from the tutorial GDS + stack JSON asset.


In [ ]:
from pathlib import Path
from semantic_geometry_builder import (
    SemanticGeometryBuilder,
    build_gds_stack_geometry_input,
)

GEOMETRY_ID = "resonator"
ROUTE = "C"
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent
ASSET_DIR = ROOT / "tutorials" / "assets"
RUN_FOLDER = ROOT / "tutorials" / "runs" / GEOMETRY_ID / f"route_{ROUTE.lower()}"
GDS_FILE = ASSET_DIR / f"{GEOMETRY_ID}.gds"
STACK_FILE = ASSET_DIR / f"{GEOMETRY_ID}.stack.json"

build_input = build_gds_stack_geometry_input(
    gds_file=GDS_FILE,
    stack_file=STACK_FILE,
)
groups = SemanticGeometryBuilder().build(
    build_input,
    route=ROUTE,
    run_folder=RUN_FOLDER,
)

xao_path = RUN_FOLDER / "geometry" / f"semantic_geometry_route_{ROUTE.lower()}.xao"
metadata_dir = RUN_FOLDER / "metadata" / "semantic_geometry"
assert xao_path.is_file(), xao_path
assert not list(RUN_FOLDER.rglob("*.msh"))
for stage in (
    "01_validate_geometry_input.json",
    "02_build_route_construction_plan.json",
    "03_build_occ_geometry.json",
    "04_export_physical_groups.json",
):
    assert (metadata_dir / stage).is_file(), stage
assert all(group.entity_tags for group in groups)

print(f"XAO: {xao_path}")
print("Physical groups:")
for group in groups:
    print(f"- dim={group.dimension} name={group.physical_name} tags={group.entity_tags}")
